# 一图知卡 (Yitu Zhika) - Kaggle训练Notebook

基于近红外光谱预测的实时食物热量分析平台

## 训练流程
1. 安装依赖
2. 挂载数据集
3. 训练阶段一 (NIR生成器)
4. 训练阶段二 (多任务网络)
5. 消融实验
6. 保存权重到Kaggle Output

## 1. 安装依赖

In [ ]:
# 安装项目依赖
!pip install -q torch torchvision torchaudio
!pip install -q timm open-clip-torch transformers
!pip install -q h5py pyyaml tensorboard scipy pulp
!pip install -q torchmetrics scikit-learn

print('依赖安装完成')

In [ ]:
# 复制项目代码（如果从Kaggle Dataset挂载）
import os
import sys

# 设置项目根目录
PROJECT_ROOT = '/kaggle/working/yitu-zhika'
os.makedirs(PROJECT_ROOT, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)

# 如果代码已作为Kaggle Dataset上传，则复制到工作目录
code_dataset = '/kaggle/input/yitu-zhika-code'
if os.path.exists(code_dataset):
    !cp -r {code_dataset}/* {PROJECT_ROOT}/
    print(f'代码已从 {code_dataset} 复制到 {PROJECT_ROOT}')
else:
    print('请在Kaggle中上传项目代码作为Dataset')

print(f'项目目录: {PROJECT_ROOT}')
print(f'Python路径: {sys.path[:3]}')

## 2. 挂载数据集

In [ ]:
# 数据集路径检测
HSIFOODINGR_PATH = '/kaggle/input/hsifoodingr64/HSIFoodIngr-64'
NUTRITION5K_PATH = '/kaggle/input/nutrition5k/Nutrition5k'

print('=== 数据集检测 ===')
for name, path in [('HSIFoodIngr-64', HSIFOODINGR_PATH), ('Nutrition5k', NUTRITION5K_PATH)]:
    exists = os.path.exists(path)
    print(f'{name}: {"✓ 存在" if exists else "✗ 未找到"} ({path})')
    if exists:
        # 显示目录内容
        items = os.listdir(path)[:5]
        print(f'  内容: {items}')

# GPU信息
import torch
print(f'\n=== GPU信息 ===')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('GPU不可用，将使用CPU训练')

## 3. 训练阶段一: NIR图像生成器

In [ ]:
import yaml
import torch

# 加载Kaggle配置
config_path = os.path.join(PROJECT_ROOT, 'configs', 'kaggle.yaml')
if not os.path.exists(config_path):
    config_path = os.path.join(PROJECT_ROOT, 'configs', 'default.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# 覆盖数据路径
config['data']['hsifoodingr_path'] = HSIFOODINGR_PATH
config['data']['nutrition5k_path'] = NUTRITION5K_PATH
config['output_dir'] = '/kaggle/working/output'
config['checkpoint_dir'] = '/kaggle/working/checkpoints'
config['log_dir'] = '/kaggle/working/logs'

print('配置加载完成')
print(f'生成器配置: {config.get("generator", {})}')

In [ ]:
# 导入训练模块
from models.generator.pix2pix import Pix2PixModel
from models.generator.losses import GeneratorLoss
from data.hsifoodingr_loader import create_hsifoodingr_dataloader

# 创建数据加载器
gen_cfg = config.get('generator', {})
data_cfg = config.get('data', {})

train_loader = create_hsifoodingr_dataloader(
    root_dir=HSIFOODINGR_PATH,
    split='train',
    img_size=data_cfg.get('img_size', 256),
    batch_size=gen_cfg.get('batch_size', 8),
    num_workers=2,
    augmentation=True,
)

val_loader = create_hsifoodingr_dataloader(
    root_dir=HSIFOODINGR_PATH,
    split='val',
    img_size=data_cfg.get('img_size', 256),
    batch_size=gen_cfg.get('batch_size', 8),
    num_workers=2,
    augmentation=False,
)

print(f'训练集: {len(train_loader.dataset)} 样本')
print(f'验证集: {len(val_loader.dataset)} 样本')

In [ ]:
# 训练阶段一
from training.train_generator import train_generator

print('开始训练NIR生成器...')
train_generator(config, resume=False)
print('阶段一训练完成!')

## 4. 训练阶段二: 多任务营养估计网络

In [ ]:
# 查找阶段一最佳模型
import glob

gen_ckpt_dir = '/kaggle/working/checkpoints/generator'
best_model_path = os.path.join(gen_ckpt_dir, 'best_model.pt')

if not os.path.exists(best_model_path):
    # 查找最新checkpoint
    ckpts = glob.glob(os.path.join(gen_ckpt_dir, '*.pt'))
    if ckpts:
        best_model_path = max(ckpts, key=os.path.getmtime)
    else:
        best_model_path = None
        print('警告: 未找到生成器checkpoint')

print(f'使用生成器: {best_model_path}')

In [ ]:
# 训练阶段二
from training.train_multitask import train_multitask

print('开始训练多任务网络...')
train_multitask(config, resume=False, generator_ckpt=best_model_path)
print('阶段二训练完成!')

## 5. 消融实验

In [ ]:
# 运行消融实验
from evaluation.ablation import AblationRunner

runner = AblationRunner(
    config_path=config_path,
    output_dir='/kaggle/working/output/ablation',
)

runner.run_all()
print('消融实验完成!')

## 6. 保存权重到Kaggle Output

In [ ]:
# 列出所有保存的权重
output_dir = '/kaggle/working'
print('=== 已保存的文件 ===')

for root, dirs, files in os.walk(output_dir):
    # 跳过隐藏目录
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        if f.endswith('.pt') or f.endswith('.json'):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  {fpath} ({size_mb:.1f} MB)')

print('\n训练完成! 权重已保存到 /kaggle/working/')
print('可在Kaggle Output中下载。')